In [ ]:
!pip install pettingzoo[mpe] torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import warnings
import os

# Filter PettingZoo deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from pettingzoo.mpe import simple_tag_v3

# ==========================================
# 1. REPLAY BUFFER
# ==========================================
class MultiAgentReplayBuffer:
    def __init__(self, max_size, n_agents, obs_dims, act_dims):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.n_agents = n_agents
        self.actor_state_memory = []
        self.actor_new_state_memory = []
        self.actor_action_memory = []

        for i in range(n_agents):
            self.actor_state_memory.append(np.zeros((self.mem_size, obs_dims[i])))
            self.actor_new_state_memory.append(np.zeros((self.mem_size, obs_dims[i])))
            self.actor_action_memory.append(np.zeros((self.mem_size, act_dims[i])))

        self.reward_memory = np.zeros((self.mem_size, n_agents))
        self.terminal_memory = np.zeros((self.mem_size, n_agents), dtype=bool)

    def store_transition(self, raw_obs, state, actions, rewards, raw_obs_, state_, dones):
        index = self.mem_cntr % self.mem_size
        for agent_idx in range(self.n_agents):
            self.actor_state_memory[agent_idx][index] = raw_obs[agent_idx]
            self.actor_new_state_memory[agent_idx][index] = raw_obs_[agent_idx]
            self.actor_action_memory[agent_idx][index] = actions[agent_idx]

        self.reward_memory[index] = rewards
        self.terminal_memory[index] = dones
        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, batch_size, replace=False)

        states, states_, actions = [], [], []
        for i in range(self.n_agents):
            states.append(self.actor_state_memory[i][batch])
            states_.append(self.actor_new_state_memory[i][batch])
            actions.append(self.actor_action_memory[i][batch])

        return states, actions, self.reward_memory[batch], states_, self.terminal_memory[batch]

    def ready(self, batch_size):
        return self.mem_cntr >= batch_size

# ==========================================
# 2. NETWORKS
# ==========================================
class CriticNetwork(nn.Module):
    def __init__(self, beta, input_dims, fc1_dims, fc2_dims, n_agents, action_dim):
        super(CriticNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dims + n_agents * action_dim, fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.q = nn.Linear(fc2_dims, 1)
        self.optimizer = optim.Adam(self.parameters(), lr=beta)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.q(x)

class ActorNetwork(nn.Module):
    def __init__(self, alpha, input_dims, fc1_dims, fc2_dims, n_actions):
        super(ActorNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dims, fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.pi = nn.Linear(fc2_dims, n_actions)
        self.optimizer = optim.Adam(self.parameters(), lr=alpha)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return torch.softmax(self.pi(x), dim=1)

# ==========================================
# 3. AGENT
# ==========================================
class Agent:
    def __init__(self, actor_dims, critic_dims, n_actions, n_agents, agent_idx,
                 alpha=0.01, beta=0.01, fc1=64, fc2=64, gamma=0.95, tau=0.01):
        self.gamma = gamma
        self.tau = tau
        self.agent_name = f'agent_{agent_idx}'

        self.actor = ActorNetwork(alpha, actor_dims, fc1, fc2, n_actions)
        self.target_actor = ActorNetwork(alpha, actor_dims, fc1, fc2, n_actions)
        self.critic = CriticNetwork(beta, critic_dims, fc1, fc2, n_agents, n_actions)
        self.target_critic = CriticNetwork(beta, critic_dims, fc1, fc2, n_agents, n_actions)

        self.update_network_parameters(tau=1)

    def choose_action(self, observation):

        device = self.actor.fc1.weight.device
        state = torch.tensor(np.array([observation]), dtype=torch.float).to(device)

        with torch.no_grad():
            actions = self.actor.forward(state)

        # Move back to CPU for numpy interaction
        return actions.detach().cpu().numpy()[0]

    def update_network_parameters(self, tau=None):
        if tau is None: tau = self.tau
        for target, local in zip(self.target_actor.parameters(), self.actor.parameters()):
            target.data.copy_(tau*local.data + (1.0-tau)*target.data)
        for target, local in zip(self.target_critic.parameters(), self.critic.parameters()):
            target.data.copy_(tau*local.data + (1.0-tau)*target.data)

# ==========================================
# 4. MADDPG SYSTEM
# ==========================================
class MADDPG:
    def __init__(self, actor_dims, critic_dims, n_agents, n_actions,
                 alpha=0.01, beta=0.01, fc1=64, fc2=64,
                 gamma=0.99, tau=0.01):
        self.agents = []
        self.n_agents = n_agents
        self.gamma = gamma
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"MADDPG initialized on device: {self.device}")

        for agent_idx in range(self.n_agents):
            agent = Agent(actor_dims[agent_idx], critic_dims, n_actions, n_agents,
                          agent_idx, alpha=alpha, beta=beta, fc1=fc1, fc2=fc2,
                          gamma=gamma, tau=tau)
            # Move models to GPU
            agent.actor.to(self.device)
            agent.target_actor.to(self.device)
            agent.critic.to(self.device)
            agent.target_critic.to(self.device)
            self.agents.append(agent)

    def choose_action(self, raw_obs):
        actions = []
        for agent_idx, agent in enumerate(self.agents):
            action = agent.choose_action(raw_obs[agent_idx])
            actions.append(action)
        return actions

    def learn(self, memory):
        BATCH_SIZE = 1024
        if not memory.ready(BATCH_SIZE): return

        actor_states, states, rewards, actor_new_states, dones = memory.sample_buffer(BATCH_SIZE)

        actor_states = [torch.tensor(actor_states[i], dtype=torch.float).to(self.device) for i in range(self.n_agents)]
        actor_new_states = [torch.tensor(actor_new_states[i], dtype=torch.float).to(self.device) for i in range(self.n_agents)]
        actions = [torch.tensor(states[i], dtype=torch.float).to(self.device) for i in range(self.n_agents)]
        rewards = torch.tensor(rewards, dtype=torch.float).to(self.device)
        dones = torch.tensor(dones).to(self.device)

        all_states = torch.cat(actor_states, dim=1)
        all_new_states = torch.cat(actor_new_states, dim=1)
        all_actions = torch.cat(actions, dim=1)

        for agent_idx, agent in enumerate(self.agents):
            new_actions = []
            with torch.no_grad():
                for i, other in enumerate(self.agents):
                    new_actions.append(other.target_actor.forward(actor_new_states[i]))
                all_new_actions = torch.cat(new_actions, dim=1)

                critic_val = agent.target_critic.forward(all_new_states, all_new_actions).flatten()
                critic_val[dones[:, agent_idx]] = 0.0
                target = rewards[:, agent_idx] + agent.gamma * critic_val

            q_expected = agent.critic.forward(all_states, all_actions).flatten()
            critic_loss = F.mse_loss(q_expected, target)

            agent.critic.optimizer.zero_grad()
            critic_loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.critic.parameters(), 1.0)
            agent.critic.optimizer.step()

            mu_states = []
            for i, other in enumerate(self.agents):
                if i == agent_idx:
                    mu_states.append(other.actor.forward(actor_states[i]))
                else:
                    mu_states.append(other.actor.forward(actor_states[i]).detach())

            actor_loss = -agent.critic.forward(all_states, torch.cat(mu_states, dim=1)).mean()

            agent.actor.optimizer.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.actor.parameters(), 1.0)
            agent.actor.optimizer.step()

            agent.update_network_parameters()

# ==========================================
# 5. MAIN LOOP
# ==========================================
def get_state_vector(obs_dict, agent_names, actor_dims):
    state = []
    for i, agent in enumerate(agent_names):
        if agent in obs_dict:
            state.append(obs_dict[agent])
        else:
            state.append(np.zeros(actor_dims[i]))
    return state

def plot_learning_curve(scores):
    running_avg = [np.mean(scores[max(0, i-100):(i+1)]) for i in range(len(scores))]
    plt.figure(figsize=(10,6))
    plt.plot(scores, alpha=0.3, color='blue', label='Raw Score')
    plt.plot(running_avg, color='red', label='Moving Avg (100)')
    plt.title("Part B: MADDPG Learning Curve (Simple Tag)")
    plt.xlabel("Episodes")
    plt.ylabel("Total Reward")
    plt.legend()
    plt.grid(True)
    plt.savefig("maddpg_learning_curve.png")
    plt.show()

if __name__ == '__main__':
    env = simple_tag_v3.parallel_env(render_mode=None, max_cycles=25, continuous_actions=False)
    env.reset()

    agent_names = env.agents
    n_agents = len(agent_names)
    actor_dims = []
    for agent in agent_names:
        actor_dims.append(env.observation_space(agent).shape[0])

    critic_dims = sum(actor_dims)
    n_actions = env.action_space(agent_names[0]).n

    # INIT MADDPG
    maddpg_agents = MADDPG(actor_dims, critic_dims, n_agents, n_actions)
    memory = MultiAgentReplayBuffer(1000000, n_agents, actor_dims, [n_actions]*n_agents)

    MAX_GAMES = 10000
    score_history = []

    print(f"Starting training on Simple Tag with {n_agents} agents...")

    for i in range(MAX_GAMES):
        obs_dict, _ = env.reset()
        score = 0
        total_steps = 0

        while env.agents:
            obs_list = get_state_vector(obs_dict, agent_names, actor_dims)
            actions_probs = maddpg_agents.choose_action(obs_list)

            actions_dict = {}
            for idx, agent in enumerate(agent_names):
                if agent in env.agents:
                    action_index = np.argmax(actions_probs[idx])
                    actions_dict[agent] = action_index

            obs_dict_, rewards_dict, terminations, truncations, _ = env.step(actions_dict)

            obs_list_ = get_state_vector(obs_dict_, agent_names, actor_dims)
            rewards = [rewards_dict.get(a, 0) for a in agent_names]
            dones_list = [terminations.get(a, False) or truncations.get(a, False) for a in agent_names]

            memory.store_transition(obs_list, obs_list, actions_probs, rewards,
                                    obs_list_, obs_list_, dones_list)

            if total_steps % 100 == 0:
                maddpg_agents.learn(memory)

            obs_dict = obs_dict_
            score += sum(rewards)
            total_steps += 1

            if all(dones_list): break

        score_history.append(score)
        if i % 10 == 0:
            avg_score = np.mean(score_history[-100:])
            print(f"Episode {i} | Avg Score: {avg_score:.1f} | Steps: {total_steps}")

    np.save("maddpg_scores.npy", np.array(score_history))
    plot_learning_curve(score_history)